In [1]:
!pip install dagshub mlflow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 7.3 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 74.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 75.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 42.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [2]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import dagshub
from mlflow.tracking import MlflowClient
import warnings
warnings.filterwarnings('ignore')

dagshub.init(
    repo_owner="sophyrise",
    repo_name='ieee-cis-fraud-detection',
    mlflow=True
)

client = MlflowClient()
print("✅ MLflow tracking URI:", mlflow.get_tracking_uri())

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=0a52cd29-f7c1-4d56-ba14-1a8c3ebdf166&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=74a8cfadeec1d0beb80bdf6ceb19726700517bbabb5f66eff7ab9b10121daf2d




Accessing as sophyrise

Initialized MLflow to track repo "sophyrise/ieee-cis-fraud-detection"

Repository sophyrise/ieee-cis-fraud-detection initialized!

✅ MLflow tracking URI: https://dagshub.com/sophyrise/ieee-cis-fraud-detection.mlflow


In [3]:
import gc

DATA_DIR = "/kaggle/input/competitions/ieee-fraud-detection"

test_txn = pd.read_csv(f"{DATA_DIR}/test_transaction.csv")
test_id  = pd.read_csv(f"{DATA_DIR}/test_identity.csv")
test_id.columns = [c.replace('-', '_') for c in test_id.columns]

test = test_txn.merge(test_id, on="TransactionID", how="left")
transaction_ids = test["TransactionID"].copy()
X_test_raw = test.drop(columns=["TransactionID"])

del test, test_txn, test_id
gc.collect()

print(f"Test shape: {X_test_raw.shape}")
print(f"Transaction IDs: {len(transaction_ids)}")

Test shape: (506691, 432)
Transaction IDs: 506691


In [ ]:
experiments = {
    "LogisticRegression-Training": "LR_Final_Pipeline",
    "Decision Trees":              "DT_Final_Pipeline",
    "Random Forest":               "RF_Final_Pipeline",
    "XGBoost":                     "XGB_Final_Pipeline",
    "Neural Networks":             "MLP_Final_Pipeline",
}

results = []
for exp_name, run_name in experiments.items():
    exp = client.get_experiment_by_name(exp_name)
    if exp is None:
        print(f"  ⚠️ not found: {exp_name}")
        continue

    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string=f"tags.mlflow.runName = '{run_name}'",
        order_by=["start_time DESC"],
        max_results=1
    )
    if not runs:
        print(f"  ⚠️ run not found: {run_name}")
        continue

    run = runs[0]
    auc = run.data.metrics.get("cv_mean_auc") or run.data.metrics.get("val_auc")
    metric_used = "cv_mean_auc" if "cv_mean_auc" in run.data.metrics else "val_auc"

    results.append({
        "model":        exp_name,
        "run_id":       run.info.run_id,
        "auc":          auc,
        "metric_used":  metric_used
    })
    print(f"  {exp_name:<35} | AUC: {auc:.4f} ({metric_used})")

results_df = pd.DataFrame(results).sort_values("auc", ascending=False).reset_index(drop=True)
print("\n📊 Model Ranking:")
print(results_df[["model", "auc", "metric_used"]].to_string())

best = results_df.iloc[0]
print(f"\n✅ Best model: {best['model']}")
print(f"   AUC: {best['auc']:.4f} ({best['metric_used']})")
print(f"   Run ID: {best['run_id']}")

In [ ]:
run = client.get_run(best['run_id'])
artifact_uri = run.info.artifact_uri

artifact_paths = {
    "LogisticRegression-Training": "logistic_regression_pipeline",
    "Decision Trees":              "decision_tree_pipeline",
    "Random Forest":               "random_forest_pipeline",
    "XGBoost":                     "xgboost_pipeline",
    "Neural Networks":             "mlp_pipeline",
}

model_path = f"{artifact_uri}/{artifact_paths[best['model']]}"
print(f"Loading model from: {model_path}")

best_model = mlflow.sklearn.load_model(model_path)
print(f"✅ Model loaded: {type(best_model.named_steps['clf']).__name__}")

In [ ]:
print("Generating predictions...")
predictions = best_model.predict_proba(X_test_raw)[:, 1]
print(f"Predictions shape: {predictions.shape}")
print(f"Prediction range: [{predictions.min():.4f}, {predictions.max():.4f}]")
print(f"Mean fraud probability: {predictions.mean():.4f}")

In [ ]:
submission = pd.DataFrame({
    "TransactionID": transaction_ids,
    "isFraud":       predictions
})

submission.to_csv("/kaggle/working/submission.csv", index=False)
print(f"✅ Submission saved: {submission.shape}")
print(submission.head(10).to_string())

In [ ]:
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

print(f"Sample submission shape: {sample_sub.shape}")
print(f"Our submission shape:    {submission.shape}")

assert len(submission) == len(sample_sub), "❌ Row count mismatch!"
assert list(submission.columns) == list(sample_sub.columns), "❌ Column mismatch!"
assert submission["isFraud"].between(0, 1).all(), "❌ Predictions out of [0,1] range!"

print("✅ Submission format is valid — ready to upload to Kaggle!")